# Week 2 In-Class Exercise: End-to-End Machine Learning Project

In the textbook (Chapter 2), we built an end-to-end ML pipeline to predict **median house values** in California.

In this exercise, you'll build a similar pipeline to answer:

> **Can we predict a country's life expectancy from its healthcare spending, economic, and demographic indicators?**

You'll follow the same workflow:
1. Get the Data
2. Explore the Data (EDA)
3. Prepare the Data for ML
4. Select and Train Models
5. Fine-Tune with Cross-Validation
6. Evaluate on Test Set

**Data source:** [World Bank Open Data](https://data.worldbank.org/)

### API References (scikit-learn 1.4)

| Class/Function | Documentation |
|----------------|---------------|
| `train_test_split` | [sklearn.model_selection.train_test_split](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.train_test_split.html) |
| `SimpleImputer` | [sklearn.impute.SimpleImputer](https://scikit-learn.org/1.4/modules/generated/sklearn.impute.SimpleImputer.html) |
| `StandardScaler` | [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/1.4/modules/generated/sklearn.preprocessing.StandardScaler.html) |
| `Pipeline` | [sklearn.pipeline.Pipeline](https://scikit-learn.org/1.4/modules/generated/sklearn.pipeline.Pipeline.html) |
| `make_pipeline` | [sklearn.pipeline.make_pipeline](https://scikit-learn.org/1.4/modules/generated/sklearn.pipeline.make_pipeline.html) |
| `ColumnTransformer` | [sklearn.compose.ColumnTransformer](https://scikit-learn.org/1.4/modules/generated/sklearn.compose.ColumnTransformer.html) |
| `LinearRegression` | [sklearn.linear_model.LinearRegression](https://scikit-learn.org/1.4/modules/generated/sklearn.linear_model.LinearRegression.html) |
| `DecisionTreeRegressor` | [sklearn.tree.DecisionTreeRegressor](https://scikit-learn.org/1.4/modules/generated/sklearn.tree.DecisionTreeRegressor.html) |
| `RandomForestRegressor` | [sklearn.ensemble.RandomForestRegressor](https://scikit-learn.org/1.4/modules/generated/sklearn.ensemble.RandomForestRegressor.html) |
| `cross_val_score` | [sklearn.model_selection.cross_val_score](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.cross_val_score.html) |
| `GridSearchCV` | [sklearn.model_selection.GridSearchCV](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.GridSearchCV.html) |
| `mean_squared_error` | [sklearn.metrics.mean_squared_error](https://scikit-learn.org/1.4/modules/generated/sklearn.metrics.mean_squared_error.html) |
| Pandas DataFrame | [pandas.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) |
| NumPy | [numpy.org/doc](https://numpy.org/doc/stable/reference/) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week2/week2_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

Run this cell to import all necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

plt.rc('font', size=12)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## Step 1: Get the Data

We fetch country-level indicators from the World Bank API:
- **SP.DYN.LE00.IN** — Life expectancy at birth (TARGET)
- **SH.XPD.CHEX.PC.CD** — Health expenditure per capita (US$)
- **NY.GDP.PCAP.CD** — GDP per capita (US$)
- **SP.POP.TOTL** — Total population
- **SE.XPD.TOTL.GD.ZS** — Government expenditure on education (% GDP)
- **SH.MED.PHYS.ZS** — Physicians per 1,000 people

This cell is provided — just run it.

In [ ]:
import urllib.request
import json

def fetch_world_bank_indicator(indicator, date_range="2018:2022"):
    """Fetch most recent value per country from World Bank API."""
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}?date={date_range}&format=json&per_page=2000"
    with urllib.request.urlopen(url) as resp:
        data = json.loads(resp.read())
    # Get country list to filter aggregates
    url_c = "https://api.worldbank.org/v2/country?per_page=400&format=json"
    with urllib.request.urlopen(url_c) as resp:
        countries = json.loads(resp.read())
    country_codes = {c['id'] for c in countries[1] if c['region']['id'] != 'NA'}
    # Most recent non-null value per country
    result = {}
    for record in data[1]:
        code = record['countryiso3code']
        if record['value'] is not None and code in country_codes and code not in result:
            result[code] = {'country': record['country']['value'], 'value': record['value']}
    return result

print("Fetching World Bank data (this may take a moment)...")
indicators = {
    'life_expectancy': 'SP.DYN.LE00.IN',
    'health_spending_pc': 'SH.XPD.CHEX.PC.CD',
    'gdp_per_capita': 'NY.GDP.PCAP.CD',
    'population': 'SP.POP.TOTL',
    'edu_spending_pct_gdp': 'SE.XPD.TOTL.GD.ZS',
    'physicians_per_1000': 'SH.MED.PHYS.ZS',
}

raw_data = {}
for name, code in indicators.items():
    print(f"  Fetching {name}...")
    raw_data[name] = fetch_world_bank_indicator(code)

# Merge into a single DataFrame
all_codes = set()
for d in raw_data.values():
    all_codes.update(d.keys())

rows = []
for code in all_codes:
    if code in raw_data['life_expectancy'] and code in raw_data['health_spending_pc']:
        row = {'country': raw_data['life_expectancy'][code]['country']}
        for name in indicators:
            row[name] = raw_data[name][code]['value'] if code in raw_data[name] else None
        rows.append(row)

df = pd.DataFrame(rows)
print(f"\nDataset loaded: {len(df)} countries, {len(df.columns)} features")
df.head()

## Step 2: Explore the Data

Let's examine the dataset — run these cells to understand the data.

In [ ]:
df.info()
print("\n")
df.describe()

In [ ]:
df.hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
# Log-transform spending for better visualization
df['log_health_spending'] = np.log(df['health_spending_pc'].clip(lower=1))
df['log_gdp_pc'] = np.log(df['gdp_per_capita'].clip(lower=1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['log_health_spending'], df['life_expectancy'], alpha=0.6)
axes[0].set_xlabel('Log(Health Spending per Capita)')
axes[0].set_ylabel('Life Expectancy (years)')
axes[0].set_title('Health Spending vs Life Expectancy')

axes[1].scatter(df['log_gdp_pc'], df['life_expectancy'], alpha=0.6)
axes[1].set_xlabel('Log(GDP per Capita)')
axes[1].set_ylabel('Life Expectancy (years)')
axes[1].set_title('GDP vs Life Expectancy')
plt.tight_layout()
plt.show()

## Step 3: Prepare the Data

### 3a: Separate features and target

In [ ]:
# Drop the country name and any log-transformed columns
feature_cols = ['health_spending_pc', 'gdp_per_capita', 'population', 
                'edu_spending_pct_gdp', 'physicians_per_1000']
target_col = 'life_expectancy'

X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nMissing values per feature:")
print(X.isnull().sum())

### 3b: Split into train and test sets

**Your turn!** Split the data into training (80%) and test (20%) sets.

**Hint:** From the textbook:
```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

API reference: [train_test_split](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:
# TODO: Split the data into train and test sets (80/20 split, random_state=42)
# Your code here:





### 3c: Build a preprocessing pipeline

**Your turn!** Create a pipeline that:
1. Imputes missing values with the median (using `SimpleImputer`)
2. Scales features with `StandardScaler`

**Hint:** From the textbook:
```python
from sklearn.pipeline import make_pipeline
num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
```

API references: [SimpleImputer](https://scikit-learn.org/1.4/modules/generated/sklearn.impute.SimpleImputer.html) | [StandardScaler](https://scikit-learn.org/1.4/modules/generated/sklearn.preprocessing.StandardScaler.html) | [make_pipeline](https://scikit-learn.org/1.4/modules/generated/sklearn.pipeline.make_pipeline.html)

In [ ]:
# TODO: Create a preprocessing pipeline with SimpleImputer and StandardScaler
# Your code here:





## Step 4: Select and Train Models

**Your turn!** Create full pipelines (preprocessing + model) for:
1. `LinearRegression`
2. `DecisionTreeRegressor`
3. `RandomForestRegressor`

Train each on the training data and compute the training RMSE.

**Hint:** From the textbook:
```python
full_pipeline = make_pipeline(preprocessing, LinearRegression())
full_pipeline.fit(X_train, y_train)
predictions = full_pipeline.predict(X_train)
rmse = root_mean_squared_error(y_train, predictions)
```

API references: [LinearRegression](https://scikit-learn.org/1.4/modules/generated/sklearn.linear_model.LinearRegression.html) | [DecisionTreeRegressor](https://scikit-learn.org/1.4/modules/generated/sklearn.tree.DecisionTreeRegressor.html) | [RandomForestRegressor](https://scikit-learn.org/1.4/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

In [ ]:
# TODO: Create and train a Linear Regression pipeline, compute training RMSE
# Your code here:





In [ ]:
# TODO: Create and train a Decision Tree pipeline, compute training RMSE
# Your code here:





In [ ]:
# TODO: Create and train a Random Forest pipeline, compute training RMSE
# Your code here:





**Question:** Which model has the lowest training RMSE? Is that necessarily the best model? Why or why not?

*Your answer:*



## Step 5: Fine-Tune with Cross-Validation

**Your turn!** Use `cross_val_score` to evaluate each model with 5-fold cross-validation.

**Hint:** From the textbook:
```python
from sklearn.model_selection import cross_val_score
scores = -cross_val_score(pipeline, X_train, y_train,
                          scoring="neg_root_mean_squared_error", cv=5)
```

API reference: [cross_val_score](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.cross_val_score.html)

In [ ]:
# TODO: Evaluate all three models with 5-fold cross-validation
# Print mean and std of RMSE for each
# Your code here:





**Question:** How do the cross-validation scores compare to the training scores? What does this tell you about overfitting?

*Your answer:*



## Step 6: Hyperparameter Tuning with GridSearchCV

**Your turn!** Use `GridSearchCV` to tune the Random Forest's `n_estimators` and `max_features`.

**Hint:**
```python
param_grid = [
    {'randomforestregressor__n_estimators': [50, 100, 200],
     'randomforestregressor__max_features': [2, 3, 4]}
]
grid_search = GridSearchCV(pipeline, param_grid, cv=5,
                           scoring='neg_root_mean_squared_error')
grid_search.fit(X_train, y_train)
```

API reference: [GridSearchCV](https://scikit-learn.org/1.4/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [ ]:
# TODO: Run GridSearchCV on the Random Forest pipeline
# Print best parameters and best score
# Your code here:





## Step 7: Evaluate on the Test Set

**Your turn!** Take the best model from GridSearchCV and evaluate it on the test set.

**Hint:**
```python
final_model = grid_search.best_estimator_
final_predictions = final_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_predictions)
```

In [ ]:
# TODO: Evaluate the best model on the test set
# Your code here:





## Reflection Questions

Answer each question in the cell below it (2-3 sentences each).

**Q1:** Why did we use a log transformation for visualization but not in the actual pipeline? Could adding log-transformed features improve the model?

*Your answer:*



**Q2:** The Decision Tree likely achieved 0 training RMSE but poor cross-validation RMSE. Explain why, using the concepts of overfitting and underfitting.

*Your answer:*



**Q3:** Why is cross-validation essential when comparing models? What would happen if we just used training RMSE?

*Your answer:*



**Q4:** What additional features or data sources could improve predictions? Name at least two and explain why.

*Your answer:*

